In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import numpy as np, glob
from PIL import Image
rng = np.random.default_rng(0)
def load(p): return np.array(Image.open(p).convert('RGB')).astype(float)

BASE = '/kaggle/input/datasets/aratisutar/phasea-run3'
K = 6
NAMES = ['FFHQ faces','CelebA portraits','Animal faces',
         'Painted portraits','Single objects','Landscapes']

def load_level(lvl, n=10, k=K):
    S, T = [], []
    for i in range(n):
        fs = sorted(glob.glob(f'{BASE}/P_L{lvl}_{i:05d}/*.png'))[:k]
        if len(fs) < k: continue
        S.append(np.stack([load(f) for f in fs]))
        T.append(load(f'{BASE}/truth_L{lvl}_{i:05d}.png'))
    return np.stack(S), np.stack(T)

LVL = {l: load_level(l) for l in range(6)}
print('loaded', [LVL[l][0].shape[0] for l in range(6)])

# ---- signals available at inference ----
def signals(S1):
    k = len(S1)
    mu = S1.mean(axis=0); sd = S1.std(axis=0).mean(axis=2)
    spread = np.mean([np.abs(S1[a]-S1[b]).mean() for a in range(k) for b in range(a+1,k)])
    flat = sd.ravel()
    conc = np.quantile(flat, 0.9)/(flat.mean()+1e-8)
    g = np.abs(np.gradient(mu.mean(axis=2))[0]) + np.abs(np.gradient(mu.mean(axis=2))[1])
    hi = g >= np.quantile(g, 0.9)
    edge = sd[hi].mean()/(sd[~hi].mean()+1e-8)
    F = np.abs(np.fft.fft2(sd))**2; N = sd.shape[0]
    fy, fx = np.meshgrid(np.fft.fftfreq(N)*N, np.fft.fftfreq(N)*N, indexing='ij')
    r = np.sqrt(fy**2+fx**2)
    lo, hiF = F[r<16].mean(), F[r>=64].mean()
    return np.array([spread, conc, edge, hiF/(lo+hiF+1e-8)])

NAMES_SIG = ['spread','concentration','edge_ratio','hf_fraction']

# ---- calibration measures ----
def rank_ustat(S1, T1, k=K):
    below = (S1 < T1[None]).sum(axis=0)
    h = np.bincount(below.ravel(), minlength=k+1).astype(float); h /= h.sum()
    return (h[0]+h[k])/(2/(k+1)) - 1

X, Y, RATIO, USTAT, LBL = [], [], [], [], []
for lvl in range(6):
    S, T = LVL[lvl]
    for S1, T1 in zip(S, T):
        k = len(S1)
        sp = np.mean([np.abs(S1[a]-S1[b]).mean() for a in range(k) for b in range(a+1,k)])
        er = np.mean([np.abs(s-T1).mean() for s in S1])
        X.append(signals(S1)); Y.append(er); RATIO.append(sp/er)
        USTAT.append(rank_ustat(S1, T1)); LBL.append(lvl)
X = np.array(X); Y = np.array(Y); RATIO = np.array(RATIO)
USTAT = np.array(USTAT); LBL = np.array(LBL)

# lambda from level 0, then per-image coverage
dev0 = np.concatenate([(np.abs(T-S.mean(axis=1))/(S.std(axis=1)+1e-8)).ravel()
                       for S, T in [LVL[0]]])
lam = float(np.quantile(dev0, 0.90))
COV = []
for lvl in range(6):
    S, T = LVL[lvl]
    for S1, T1 in zip(S, T):
        mu, sd = S1.mean(axis=0), S1.std(axis=0)+1e-8
        COV.append((np.abs(T1-mu) <= lam*sd).mean())
COV = np.array(COV)
print(f'lambda = {lam:.2f}\n')

def auc(score, label):
    pos, neg = score[label==1], score[label==0]
    w = (pos[:,None] > neg[None,:]).sum(); t = (pos[:,None] == neg[None,:]).sum()
    return (w + 0.5*t)/(len(pos)*len(neg))

labels = {
  'error (top 25%)' : (Y > np.quantile(Y, 0.75)).astype(int),
  'ratio (old)'     : (RATIO < np.quantile(RATIO, 0.25)).astype(int),
  'rank U-stat'     : (USTAT > np.quantile(USTAT, 0.75)).astype(int),
  'coverage short'  : (COV   < np.quantile(COV,   0.25)).astype(int),
}

print(f"{'signal':>15s}" + ''.join(f'{k:>18s}' for k in labels))
for j, nm in enumerate(NAMES_SIG):
    row = ''.join(f'{max(auc(X[:,j],l), 1-auc(X[:,j],l)):18.3f}' for l in labels.values())
    print(f'{nm:>15s}{row}')

print('\nlabel agreement (of 15):')
ks = list(labels)
for a_ in range(len(ks)):
    for b_ in range(a_+1, len(ks)):
        print(f'  {ks[a_]:<16s} vs {ks[b_]:<16s} {(labels[ks[a_]] & labels[ks[b_]]).sum():2d}')

loaded [10, 10, 10, 10, 10, 10]
lambda = 2.99

         signal   error (top 25%)       ratio (old)       rank U-stat    coverage short
         spread             0.926             0.505             0.587             0.567
  concentration             0.708             0.664             0.576             0.695
     edge_ratio             0.778             0.639             0.510             0.695
    hf_fraction             0.738             0.692             0.607             0.748

label agreement (of 15):
  error (top 25%)  vs ratio (old)       8
  error (top 25%)  vs rank U-stat       6
  error (top 25%)  vs coverage short    8
  ratio (old)      vs rank U-stat      13
  ratio (old)      vs coverage short   14
  rank U-stat      vs coverage short   12


In [3]:
cov_short = labels['coverage short']
B = 5000

print(f"{'signal':>15s} {'AUC':>7s} {'null med':>9s} {'null 95%':>9s} {'p':>8s}")
for j, nm in enumerate(NAMES_SIG):
    a = auc(X[:, j], cov_short)
    obs = max(a, 1-a)
    null = np.empty(B)
    for b in range(B):
        perm = rng.permutation(cov_short)
        aa = auc(X[:, j], perm)
        null[b] = max(aa, 1-aa)
    p = (null >= obs).mean()
    print(f'{nm:>15s} {obs:7.3f} {np.median(null):9.3f} '
          f'{np.quantile(null,0.95):9.3f} {p:8.4f}')

         signal     AUC  null med  null 95%        p
         spread   0.567     0.559     0.667   0.4434
  concentration   0.695     0.559     0.668   0.0218
     edge_ratio   0.695     0.557     0.668   0.0238
    hf_fraction   0.748     0.559     0.668   0.0030


In [4]:
print('WITHIN-LEVEL: do miscalibrated images score higher on hf_fraction?\n')
print(f"{'level':<20s} {'n_pos':>6s} {'AUC':>7s} {'mean+':>8s} {'mean-':>8s}")
within = []
for lvl in range(6):
    m = LBL == lvl
    lab = cov_short[m]
    if lab.sum() == 0 or lab.sum() == m.sum():
        print(f'{NAMES[lvl]:<20s} {lab.sum():6d}      -  (no contrast)')
        continue
    s = X[m, 3]
    a = auc(s, lab)
    within.append(a)
    print(f'{NAMES[lvl]:<20s} {lab.sum():6d} {a:7.3f} '
          f'{s[lab==1].mean():8.4f} {s[lab==0].mean():8.4f}')

if within:
    print(f'\nmean within-level AUC: {np.mean(within):.3f}  (0.5 = no signal)')
    print(f'levels where AUC > 0.5: {sum(a>0.5 for a in within)} of {len(within)}')

WITHIN-LEVEL: do miscalibrated images score higher on hf_fraction?

level                 n_pos     AUC    mean+    mean-
FFHQ faces                0      -  (no contrast)
CelebA portraits          0      -  (no contrast)
Animal faces              2   0.562   0.0001   0.0001
Painted portraits         7   0.381   0.0001   0.0001
Single objects            1   0.889   0.0007   0.0004
Landscapes                5   0.360   0.0002   0.0003

mean within-level AUC: 0.548  (0.5 = no signal)
levels where AUC > 0.5: 2 of 4


In [5]:
def hf_of_image(T1):
    g = T1.mean(axis=2)
    F = np.abs(np.fft.fft2(g))**2
    N = g.shape[0]
    fy, fx = np.meshgrid(np.fft.fftfreq(N)*N, np.fft.fftfreq(N)*N, indexing='ij')
    r = np.sqrt(fy**2+fx**2)
    lo, hi = F[r<16].mean(), F[r>=64].mean()
    return hi/(lo+hi+1e-8)

HF_IMG = np.array([hf_of_image(T1) for lvl in range(6) for T1 in LVL[lvl][1]])

a_sig = auc(X[:,3], cov_short); a_sig = max(a_sig, 1-a_sig)
a_img = auc(HF_IMG,  cov_short); a_img = max(a_img, 1-a_img)
print(f'hf of DISAGREEMENT map : AUC {a_sig:.3f}')
print(f'hf of GROUND-TRUTH image: AUC {a_img:.3f}')
print(f'correlation between them: {np.corrcoef(X[:,3], HF_IMG)[0,1]:+.3f}')

hf of DISAGREEMENT map : AUC 0.748
hf of GROUND-TRUTH image: AUC 0.670
correlation between them: -0.160


In [6]:
B = 5000
boot = np.empty(B)
n = len(Y)
for b in range(B):
    idx = rng.choice(n, n, replace=True)
    if cov_short[idx].sum() in (0, n): 
        boot[b] = np.nan; continue
    a = auc(X[idx,3], cov_short[idx])
    boot[b] = max(a, 1-a)
boot = boot[~np.isnan(boot)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f'hf_fraction AUC 0.748, bootstrap 95% CI [{lo:.3f}, {hi:.3f}]')

hf_fraction AUC 0.748, bootstrap 95% CI [0.599, 0.879]
